# **FUSIÓN DE AMBAS FASES - PERFILADO CON VECTORIZADO(MLP)**

In [ ]:
import joblib
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
from groq import Groq  
from datetime import datetime
import time
import matplotlib.pyplot as plt
from fpdf import FPDF
import os
from dotenv import load_dotenv

load_dotenv()
api_key_groq = os.getenv("GROQ_API_KEY")
# ==========================================
# 0. CONFIGURACIÓN DE LA API EN LA NUBE
# ==========================================
client = Groq(api_key=api_key_groq)
# ==========================================
# 1. CARGA DE CEREBROS (FASES 1 Y 2)
# ==========================================
print("Cargando cerebro del sistema (Esto puede tomar unos segundos)...")
tokenizer = AutoTokenizer.from_pretrained("pysentimiento/robertuito-base-uncased")
robertuito = AutoModel.from_pretrained("pysentimiento/robertuito-base-uncased")
mlp_afecto = joblib.load('modelos/modelo_MPL_afecto.pkl')

scaler_f1 = joblib.load('modelos/scaler_clustering.pkl')
kmeans_f1 = joblib.load('modelos/modelo_clustering_v1.pkl')

# Calculamos ID Verde y Rojo dinámicamente para la Fase 1
sumas_centros = kmeans_f1.cluster_centers_.sum(axis=1)
ID_VERDE = sumas_centros.argmin()
ID_ROJO = sumas_centros.argmax()

# ==========================================
# 2. DICCIONARIO Y FUNCIONES DE TRIAGE
# ==========================================
DICCIONARIO_CLINICO = {
    #depresion
    'PHQ9_Anhedonia': [
        'apatía', 'desmotivad', 'aburrid', 'interés', 'placer', 'hueva', 'sin ganas', 'flojera', 'nada me importa'],
    #depresion
    'PHQ9_Estado_Animo': [
        'deprimid', 'sin esperanza', 'miserable', 'triste', 'infeliz', 'agüitad', 'bajón', 'oscuridad', 'llorar', 'sol', 'vací'],
    #depresion
    'PHQ9_Somático': [
        'insomnio', 'sueño', 'dormir', 'exhaust', 'fatiga', 'energía', 'cansad', 'pesad', 'débil', 'agotamient', 'apetito', 'peso', 'comer', 'hambre', 'lent', 'no rindo', 'desvelad'],
    #depresion
    'PHQ9_Cognitivo_Autoestima': [
        'fracaso', 'inútil', 'culpa', 'aislad', 'estorbo', 'carga', 'vergüenza', 'decepcionad', 'concentración', 'atención', 'decisiones', 'enfocarme', 'olvidadiz', 'tont', 'brut', 'no sirvo'],
    #caso de ansiedad
    'GAD7_Fisico': [
        'corazón', 'pecho', 'respirar', 'aire', 'tembland', 'temblor', 'sudor', 'náusea', 'mareo', 'entumecid', 'hormigueo', 'estómago', 'ahogand', 'tensión', 'taquicardia'],
    #caso de ansiedad
    'GAD7_Mental': [
        'pánico', 'ataque', 'ansios', 'ansiedad', 'preocupación', 'miedo', 'asustad', 'pensamientos', 'sobrepensar', 'obsesiv', 'irracional', 'control', 'loc', 'angustia'],
    #estres
    'Burnout_Situacional': [
        'abrumad', 'presión', 'entrega', 'carga', 'quemad', 'burnout', 'batalland', 'colapso', 'irritable', 'cabeza', 'exigente', 'deudas', 'dinero', 'lana', 'escuela', 'universidad', 'uni', 'prepa', 'trabajo', 'chamba', 'responsabilidades', 'nervios', 'demasiado', 'gente', 'estrés', 'estresad', 'harto', 'ya no puedo'],
    #caso suicida
    # --- LISTA AUTOLÍTICA EXPANDIDA ---
    'Alerta_Suicida': [
        'matarme', 'matar', 'suicidio', 'suicidarme', 'suicidar', 
        'terminar con todo', 'terminar con mi vida','termino con mi vida','quitarme la vida', 
        'morir', 'muerte', 'adiós', 'carta', 'pastillas', 'sobredosis', 
        'puente', 'colgarme', 'mejor muert', 'sin salida', 'harto de la vida', 
        'rendirme', 'acabar con todo', 'no vale la pena', 'desaparecer', 
        'no despertar', 'ya no quiero vivir', 'dormir para siempre', 
        'cortarme', 'hacerme daño', 'dejar de existir'
    ],
    #afecto positivo
    'Afecto_Positivo': 
        ['hoy', 'personas', 'pensar', 'amig', 'familia', 'vida', 'bien', 'chido', 'feliz', 'ok', 'mejor', 'tranquil', 'calma', 'esperanza', 'motivad', 'descansé']
}

def motor_ner_definitivo(texto):
    categorias_detectadas = []
    texto_limpio = texto.lower()
    for categoria, palabras_clave in DICCIONARIO_CLINICO.items():
        if any(palabra in texto_limpio for palabra in palabras_clave):
            categorias_detectadas.append(categoria)
    return categorias_detectadas

def puente_clinico_sin_sesgo(datos_usuario, scaler_cargado, kmeans_cargado, id_verde, id_rojo):
    cols_clinicas = ['Growing_Stress', 'Coping_Struggles', 'Work_Interest', 'Mood_Swings', 'Changes_Habits', 'Social_Weakness', 'Mental_Health_History', 'Quarantine_Frustrations', 'Weight_Change']
    reg = {
        'Growing_Stress': datos_usuario.get('estres', 0), 'Coping_Struggles': datos_usuario.get('afrontamiento', 0),
        'Work_Interest': datos_usuario.get('interes', 0), 'Mood_Swings': datos_usuario.get('humor', 0),
        'Changes_Habits': datos_usuario.get('habitos', 0), 'Social_Weakness': datos_usuario.get('debilidad_social', 0),
        'Mental_Health_History': datos_usuario.get('historial', 0), 'Quarantine_Frustrations': datos_usuario.get('irritabilidad', 0),
        'Weight_Change': datos_usuario.get('peso', 0)
    }
    
    df_in = pd.DataFrame([reg], columns=cols_clinicas)
    df_in_scaled = scaler_cargado.transform(df_in)
    cluster_ia = kmeans_cargado.predict(df_in_scaled)[0]
    
    # --- CÁLCULO DE LA SUMA ---
    sintomas_base = [reg['Growing_Stress'], reg['Coping_Struggles'], reg['Work_Interest'], reg['Mood_Swings'], reg['Changes_Habits'], reg['Social_Weakness']]
    puntuacion_real = sum(sintomas_base)
    
    # --- MODO TRANSPARENCIA (EXPLAINABLE AI) ---
    print("\n" + "* "*15)
    print("   [PROCESANDO TRIAGE MATEMÁTICO - FASE 1]")
    print("*"*15)
    print(f" -> 1. Extrayendo variables clínicas: {list(reg.values())}")
    print(f" -> 2. Escalando datos (StandardScaler)...")
    print(f" -> 3. K-Means (IA) asignó el cluster: {cluster_ia} (Rojo crítico es {id_rojo})")
    print(f" -> 4. Calculando Carga Sintomatológica (Suma):")
    print(f"       {sintomas_base[0]} + {sintomas_base[1]} + {sintomas_base[2]} + {sintomas_base[3]} + {sintomas_base[4]} + {sintomas_base[5]} = {puntuacion_real}/12 puntos")
    
    # --- APLICACIÓN DE REGLAS ---
    if puntuacion_real <= 2:
        if reg['Mental_Health_History'] == 2 or cluster_ia == id_rojo: 
            print(" -> 5. Regla de Blindaje: Sube a MODERADO por historial o IA.")
            return "RIESGO MODERADO"
        print(" -> 5. Dictamen: RIESGO BAJO (Puntuación dentro de 0-2).")
        return "RIESGO BAJO"
        
    elif 3 <= puntuacion_real <= 5:
        if cluster_ia == id_rojo or reg['Mental_Health_History'] == 2: 
            print(" -> 5. Regla de Blindaje: Sube a ALTO por clúster crítico o historial.")
            return " RIESGO ALTO"
        print(" -> 5. Dictamen: RIESGO MODERADO (Puntuación dentro de 3-5).")
        return "RIESGO MODERADO"
        
    else: 
        print(" -> 5. Dictamen: RIESGO ALTO (Puntuación >= 6, umbral clínico superado).")
        return "RIESGO ALTO"

def procesar_triage_unificado(datos_cuestionario, mensaje_chat):
    riesgo_base = puente_clinico_sin_sesgo(datos_cuestionario, scaler_f1, kmeans_f1, ID_VERDE, ID_ROJO)
    
    inputs = tokenizer(mensaje_chat, return_tensors="pt", padding=True, truncation=True, max_length=128)
    with torch.no_grad(): outputs = robertuito(**inputs)
    embedding = outputs.last_hidden_state[:, 0, :].numpy()

    # ---  PROBABILIDADES ---
    probabilidades = mlp_afecto.predict_proba(embedding)[0]
    
    # np.argsort ordena de menor a mayor. Con [::-1] lo volteamos de mayor a menor.
    indices_ordenados = np.argsort(probabilidades)[::-1] 
    
    idx_principal = indices_ordenados[0] # El ganador (Top 1)
    idx_secundario = indices_ordenados[1] # La segunda opción más probable (Top 2)
    confianza = probabilidades[idx_principal]
    
    mapeo_afecto = {0: 'Normal', 1: 'Stress', 2: 'Anxiety', 3: 'Depression', 4: 'Suicidal'}
        
    estado_actual = mapeo_afecto[idx_principal]   
    
    sintomas_detectados = motor_ner_definitivo(mensaje_chat)
    
    # --- SAFETY GATE (FILTRO CRUZADO DINÁMICO) ---
    alerta_critica = False
    
    # 1. Si el NER detecta una palabra explícita de muerte, BLOQUEO TOTAL.
    if "Alerta_Suicida" in sintomas_detectados:
        alerta_critica = True
        estado_actual = "Suicidal"

    # 2. Si la IA dice Suicida pero el NER no encontró alertas...
    elif estado_actual == "Suicidal":
        # Evaluamos si es un Falso Positivo (si no está 99.8% segura)
        if confianza < 0.998:
            # Asignamos la segunda emoción más probable (Ej. Stress o Anxiety)
            estado_actual = mapeo_afecto[idx_secundario] 
            alerta_critica = False
        else:
            alerta_critica = True

    return {
        "perfil_riesgo_f1": riesgo_base, 
        "afecto_detectado_f2": estado_actual, 
        "confianza_f2": confianza, 
        "sintomas_ner": sintomas_detectados, 
        "alerta_critica": alerta_critica,
        "probabilidades_crudas": probabilidades
    }

# ==========================================
# 3. LLAMADA A LA NUBE (FASE 3 - GROQ CON MEMORIA)
# ==========================================
def interactuar_con_ia(historial_mensajes):
    intentos = 0
    max_intentos = 3
    
    while intentos < max_intentos:
    
        """
        Recibe el historial completo de la conversación
        """
        try:
            chat_completion = client.chat.completions.create(
                messages=historial_mensajes,
                model="llama-3.3-70b-versatile",
                temperature=0.7,
                max_tokens=300
            )
            return chat_completion.choices[0].message.content
        except Exception as e:
            error_str = str(e).lower()
            intentos += 1
            
            # Si es un error de "Rate Limit" (fuimos muy rápido), esperamos y reintentamos
            if "rate limit" in error_str or "429" in error_str or "too many requests" in error_str:
                print(f"  [!] Límite de velocidad de la API. Esperando 6 segundos (Intento {intentos}/{max_intentos})...")
                time.sleep(6)
            else:
                # Si es un error diferente (se fue el internet, etc.)
                return f"[ERROR_API_TOKENS] Detalles: {e}"
        return f"[ERROR_API_TOKENS] Límite de reintentos superado: {e}"

# ==========================================
# 4. APLICACIÓN INTERACTIVA (Cuestionario -> Chat)
# ==========================================
def hacer_pregunta(numero, texto):
    """Muestra un menú tipo PHQ-9 para cada pregunta clínica."""
    print(f"\n{numero}. {texto}")
    print("  [0] No / Nunca / Ningún día")
    print("  [1] A veces / Regularmente / Varios días")
    print("  [2] Sí / Casi siempre / Muy frecuente")
    while True:
        try:
            resp = int(input("  -> Selecciona una opción (0/1/2): "))
            if resp in [0, 1, 2]:
                return resp
            print("  Opción no válida. Por favor ingresa 0, 1 o 2.")
        except ValueError:
            print("  Por favor ingresa un número válido.")


def iniciar_aplicacion():
    print("\n" + "="*60)
    print("   SISTEMA DE TRIAGE CLÍNICO Y CONTENCIÓN (DEMO TÉSIS)")
    print("="*60)
    
    print("\n--- BLOQUE 1: CONTEXTO ---")
    edad = input("¿Qué edad tienes? (ej. 16-20, 21-25, etc.): ")
    ocupacion = input("Actualmente, ¿a qué te dedicas la mayor parte del tiempo? (Estudiante, Trabajo Corporativo, etc.): ")
    nombre = input("¿Cómo prefieres que me dirija a ti?")
    
    print("\n--- BLOQUE 2: TRIAGE MATEMÁTICO ---")
    estres = hacer_pregunta("1", "En las últimas semanas, ¿has sentido que el estrés te sobrepasa o te cuesta relajarte?")
    interes = hacer_pregunta("2", "¿Has perdido el interés o las ganas de hacer las cosas que antes disfrutabas?")
    afrontamiento = hacer_pregunta("3", "Cuando enfrentas problemas cotidianos, ¿sientes que te quedas sin energía o herramientas para resolverlos?")
    humor = hacer_pregunta("4", "¿Has notado cambios de humor repentinos, sintiéndote muy triste o sin esperanza?")
    debilidad_social = hacer_pregunta("5", "¿Te has sentido aislado(a) o has evitado interactuar con personas cercanas últimamente?")
    irritabilidad = hacer_pregunta("6", "¿Te has sentido fácilmente frustrado(a), irritable o con los nervios de punta?")
    habitos_peso = hacer_pregunta("7", "¿Has notado cambios fuertes en tus hábitos diarios (sueño) o cambios importantes en tu apetito/peso?")
    
    print("\n--- BLOQUE 3: MODIFICADORES CRÍTICOS ---")
    print("\n8. ¿Cuánto tiempo llevas sintiendo la necesidad de aislarte?")
    print("  [0] Salgo normal / No me aíslo")
    print("  [1] Llevo unas semanas así")
    print("  [2] Llevo más de un mes aislado(a)")
    aislamiento = int(input("  -> Selecciona una opción (0/1/2): "))
    
    print("\n9. ¿Has recibido atención psicológica o psiquiátrica previamente?")
    print("  [0] No, nunca")
    print("  [2] Sí, he estado en tratamiento")
    historial = int(input("  -> Selecciona una opción (0 o 2): "))

    datos_usuario = {
        'age_cat': edad, 'ocupacion': ocupacion, 'nombre': nombre, 'estres': estres,
        'interes': interes, 'afrontamiento': afrontamiento, 'humor': humor,
        'debilidad_social': debilidad_social, 'irritabilidad': irritabilidad,
        'habitos': habitos_peso, 'peso': habitos_peso, 'aislamiento': aislamiento, 'historial': historial
    }
    
    print("\nCuestionario guardado exitosamente. Triage de Fase 1 completado.")
    
    # --- LA PAUSA DE SEGURIDAD ---
    input("\nPresiona ENTER para entrar a la sala de chat en vivo...")
    
    print("\n--- PASO 2: CHAT EN VIVO ---")
    print("(Escribe 'salir' para terminar la conversación)")
    
    historial_chat = []
    registro_emociones = [] # Para el PDF
    riesgo_global_f1 = ""   # Para el PDF
    
    # --- CONFIGURACIÓN ---
    TURNOS_MAXIMOS = 12       # Límite clínico de la sesión
    TURNO_DE_AVISO = 10       # Momento en que la IA empieza a despedirse
    turnos_actuales = 0
    historial_probabilidades = []
    
    while True:
        #verificar el limite clinico
        if turnos_actuales >= TURNOS_MAXIMOS:
            print("\n[SISTEMA]: El protocolo de primeros auxilios psicológicos ha concluido.")
            print("IA: Nuestro tiempo de sesión ha terminado por hoy. Ha sido un paso muy valiente hablar de esto. Por favor cuídate mucho, generaré tu reporte clínico.")
            break # Salimos del chat para hacer el PDF
                    
        # 1. Pedimos el texto con un indicador discreto
        mensaje_usuario = input("\nEscribe tu mensaje y presiona Enter -> ")
        
        # Filtro de seguridad rápido para evitar vacíos
        if not mensaje_usuario.strip():
            print("\nIA: Te escucho, tómate tu tiempo. ¿Me decías?")
            continue
            
        # 2. IMPRIMIMOS el  MENSAJE EN EL OUTPUT (Formato formal)
        print(f"\n👤 TÚ: {mensaje_usuario}")
        
        if mensaje_usuario.lower() in ['salir', 'exit', 'adiós']:
            print("\nIA: Ha sido un gusto escucharte. Por favor cuídate mucho. Generando tu reporte...")
            break
        
        turnos_actuales += 1
        # Analizamos el mensaje uniendo la Fase 1 y Fase 2
        reporte = procesar_triage_unificado(datos_usuario, mensaje_usuario)
        riesgo_global_f1 = reporte['perfil_riesgo_f1'] 
        historial_probabilidades.append(reporte['probabilidades_crudas'])
        
        info_emocional = f"MLP: {reporte['afecto_detectado_f2']} (Seguridad: {reporte['confianza_f2']:.1%}) | NER: {reporte['sintomas_ner']}"
        print(f"  [DEBUG FASE 2] -> {info_emocional}")
        registro_emociones.append(info_emocional)
        # MODO DEBUG 
        print(f"  [DEBUG FASE 2] -> Emoción MLP: {reporte['afecto_detectado_f2']} (Seguridad: {reporte['confianza_f2']:.1%}) | NER: {reporte['sintomas_ner']}")  
        
              
        # VÍA A: SAFETY GATE
        if reporte['alerta_critica']:
            print("\n[ALERTA INTERNA]: Riesgo Crítico/Suicida Detectado. Activando protocolo de contención continua.")
            print("IA: Esta es una situación de emergencia médica. Por favor llama ahora mismo a la Línea de la Vida al 800 911 2000. Hay profesionales listos para ayudarte las 24 horas.")
            # En lugar de hacer 'break', inyectamos un prompt de crisis extrema para que la IA tome el control
            prompt_crisis = """
            ROL Y PROPÓSITO:
            Eres un especialista en intervención en crisis. El usuario acaba de expresar intenciones o pensamientos suicidas (RIESGO CRÍTICO).
            
            REGLAS ESTRICTAS DE COMPORTAMIENTO:
            1. ÚNICO OBJETIVO: Tu objetivo absoluto es mantener a la persona platicando contigo para anclarla al presente, validar su dolor profundo sin juzgar, y persuadirla suavemente de que busque ayuda.
            2. NÚMERO DE EMERGENCIA: En tu respuesta, de forma empática, pídele que llame a la Línea de la Vida al 800 911 2000. Diles que hay profesionales listos para ayudarles.
            3. LONGITUD Y CIERRE: Responde en máximo 4 oraciones. TERMINA SIEMPRE con una pregunta abierta que invite al usuario a seguir desahogándose contigo (ej. "¿Qué es lo que más te duele en este momento?", "¿Quieres contarme más sobre eso? Estoy aquí").
            4. NO TERMINES LA CONVERSACIÓN.
            """
            
            if len(historial_chat) == 0:
                historial_chat.append({"role": "system", "content": prompt_crisis})
            else:
                # Sobrescribimos el prompt normal con el de crisis
                historial_chat[0] = {"role": "system", "content": prompt_crisis}
                
            historial_chat.append({"role": "user", "content": mensaje_usuario})
            
            print("  ... Consultando protocolo de emergencia con la IA ...") 
            respuesta = interactuar_con_ia(historial_chat)
            
            if "[ERROR_API_TOKENS]" in respuesta:
                print("\n[ALERTA DEL SISTEMA]: Fallo de conexión en emergencia. Por favor, llama al 800 911 2000 de inmediato.")
                break
                
            print(f"IA: {respuesta}\n")
            historial_chat.append({"role": "assistant", "content": respuesta})
            
            #'continue' para que el ciclo vuelva arriba y pida el siguiente mensaje del usuario
            continue
            
        # ==========================================
        # VÍA B: LLAMADA A GROQ (CHAT NORMAL)
        # ==========================================
        instruccion = ""
        if "BAJO" in reporte['perfil_riesgo_f1']: instruccion = "Sé amigable y cálido. Enfócate en mantener su bienestar."
        elif "MODERADO" in reporte['perfil_riesgo_f1']: instruccion = "Sé cauteloso, explora su nivel de estrés y valida sus emociones."
        elif "ALTO" in reporte['perfil_riesgo_f1']: instruccion = "Ofrece contención prioritaria, sé muy delicado y sugiere fuertemente buscar apoyo psicológico."

        # Traducimos la emoción del MLP al español para que la IA la integre mejor
        diccionario_traduccion = {'Stress': 'estrés', 'Anxiety': 'ansiedad', 'Depression': 'tristeza o depresión', 'Normal': 'tranquilidad'}
        emocion_traducida = diccionario_traduccion.get(reporte['afecto_detectado_f2'], reporte['afecto_detectado_f2'])

        ##psicoeducacion basada en la confianza que debe de ser mayor del 75%
        if reporte['confianza_f2'] > 0.75 and reporte['afecto_detectado_f2'] != 'Normal':
            regla_psicoeducacion = f"2. PSICOEDUCACIÓN: Si el usuario expresa malestar, menciona sutilmente que podría estar relacionado con síntomas de {emocion_traducida}. NUNCA des un diagnóstico definitivo. SIN EMBARGO, si el usuario está notoriamente feliz, motivado o tranquilo, IGNORA la métrica clínica, no menciones síntomas y acompáñalo en su alegría y logro."
        else:
            regla_psicoeducacion = "2. PSICOEDUCACIÓN: Escucha activamente. Debido a la ambigüedad del mensaje, NO menciones diagnósticos ni síntomas específicos en este turno, solo acompaña sus emociones (sean positivas o negativas) de forma muy empática."
        
        
        prompt_sistema = f"""
        ROL Y PROPÓSITO:
        Eres un asistente de primeros auxilios psicológicos y contención emocional. NO eres un médico, psiquiatra ni psicólogo clínico. Tu objetivo es escuchar, validar emociones y brindar psicoeducación.

        CONTEXTO DEL PACIENTE:
        - Edad: {datos_usuario['age_cat']}
        - Ocupación: {datos_usuario['ocupacion']}
        - Pronombres preferidos: {datos_usuario['nombre']}
        - Riesgo Base (Fase 1): {reporte['perfil_riesgo_f1']}
        - Emoción actual detectada (Fase 2): {emocion_traducida}
        - Síntomas clínicos detectados en su mensaje: {', '.join(reporte['sintomas_ner']) if reporte['sintomas_ner'] else 'Ninguno'}

        REGLAS ESTRICTAS DE COMPORTAMIENTO:
        1. TONO: {instruccion} Sé conversacional y empático.
        2. PSICOEDUCACIÓN: Si el usuario expresa malestar, menciona sutilmente que podría estar relacionado con síntomas de {emocion_traducida},NUNCA des un diagnóstico definitivo.. SIN EMBARGO, si el usuario está notoriamente feliz, motivado o tranquilo, IGNORA la métrica clínica, no menciones síntomas y acompáñalo en su alegría y logro.
        3. DISCLAIMER: Incluye una frase natural indicando que eres una IA y no sustituyes atención médica, solo es necesario que se EXPRESE UNA SOLA VEZ ,concentrate en escuchar.
        4. LONGITUD: Responde en máximo 4 oraciones. 
        4. REGLA DE EMERGENCIA (PLAN B): Si el usuario menciona querer morir, hacer daño o despedirse, DEBES responder textualmente recomendando la Línea de la Vida en México (800 911 2000) y preguntarle qué siente en este momento.
        """
        
        if turnos_actuales >= TURNO_DE_AVISO:
            prompt_sistema += "\nINSTRUCCIÓN CRÍTICA: La sesión está por terminar. Empieza a concluir la charla de forma muy cálida, dale un consejo final y NO le hagas preguntas que alarguen la plática."
        else:
            prompt_sistema += "\nTermina siempre con una pregunta suave para invitarlo a desahogarse."
        
        if len(historial_chat) == 0:
            historial_chat.append({"role": "system", "content": prompt_sistema})
        else:
            historial_chat[0] = {"role": "system", "content": prompt_sistema}
            
        historial_chat.append({"role": "user", "content": mensaje_usuario})
        
        print("  ... Consultando a la IA ...") 
        respuesta = interactuar_con_ia(historial_chat)
        
        if "[ERROR_API_TOKENS]" in respuesta:
            print("\n[ALERTA DEL SISTEMA]: La capacidad de procesamiento de la IA ha llegado a su límite técnico por esta sesión.")
            print("IA: Disculpa, mi sistema ha alcanzado el límite de carga y debemos pausar nuestra charla. Ha sido un paso muy valiente buscar ayuda. Por favor, guarda tu reporte clínico y considera buscar apoyo con un profesional.")
            historial_chat.append({"role": "assistant", "content": "[FIN ABRUPTO] La sesión terminó por límite de tokens en la API de Groq."})
            break # Se cierra el chat por falla técnica
            
        # 3. IMPRIMIMOS LA RESPUESTA DE LA IA
        print(f"IA: {respuesta}\n")
        
        historial_chat.append({"role": "assistant", "content": respuesta})
        
    generar_reporte_y_grafica(historial_probabilidades, datos_usuario, riesgo_global_f1, datos_usuario.get('nombre', 'Paciente'))
   
def generar_reporte_y_grafica(historial_probabilidades, datos_usuario, riesgo_global, nombre_usuario="Paciente"):
    print("\n Generando gráfica y reporte clínico detallado...")
    
    carpeta_destino = "pdf"
    if not os.path.exists(carpeta_destino):
        os.makedirs(carpeta_destino)
    
    if historial_probabilidades:
        promedios = np.mean(historial_probabilidades, axis=0)
    else:
        promedios = [0, 0, 0, 0, 0]

    # 1. Generación de la gráfica PNG
    etiquetas = ['Normal', 'Estrés', 'Ansiedad', 'Depresión', 'Riesgo Suicida']
    colores = ['#4CAF50', '#FFC107', '#FF9800', '#9C27B0', '#F44336']
    plt.figure(figsize=(8, 5))
    barras = plt.bar(etiquetas, promedios * 100, color=colores)
    plt.title(f'Perfil Emocional Promedio de la Sesión (MLP)', fontsize=14)
    plt.ylabel('Porcentaje de Confianza (%)')
    plt.ylim(0, 100)
    
    for barra in barras:
        yval = barra.get_height()
        # --- CAMBIO AQUÍ: Porcentaje ADENTRO de la gráfica ---
        # Si la barra es muy bajita, lo ponemos arriba (va='bottom'), si es alta, adentro (va='top')
        posicion_y = yval - 5 if yval > 10 else yval + 1
        color_texto = 'white' if yval > 10 else 'black'
        
        plt.text(
            barra.get_x() + barra.get_width()/2, 
            posicion_y, 
            f'{yval:.1f}%', 
            ha='center', 
            va='center', 
            color=color_texto, 
            fontweight='bold'
        )

    # --- NUEVO: Ruta de la gráfica dentro de la carpeta pdf ---
    ruta_grafica = os.path.join(carpeta_destino, 'grafica_emociones.png')
    plt.tight_layout()
    plt.savefig(ruta_grafica)
    plt.close()

    # ==========================================
    # GENERACIÓN DEL REPORTE PDF PROFESIONAL
    # ==========================================
    pdf = FPDF(orientation='P', unit='mm', format='Letter')
    pdf.add_page()
    
    # ENCABEZADO TÉCNICO
    pdf.set_font('Helvetica', 'B', 16)
    pdf.cell(0, 10, 'REPORTE PSICOLÓGICO AUTOMATIZADO', new_x="LMARGIN", new_y="NEXT", align='C')
    
    pdf.set_font('Helvetica', 'B', 9)
    pdf.set_text_color(0, 51, 153) # Azul clínico
    pdf.cell(0, 5, 'Documento de apoyo para valoración profesional', new_x="LMARGIN", new_y="NEXT", align='C')
    pdf.set_text_color(0, 0, 0)
    pdf.ln(5)

    pdf.set_font('Helvetica', 'B', 11)
    pdf.cell(0, 7, f'Identificador/Nombre: {nombre_usuario}', new_x="LMARGIN", new_y="NEXT")
    pdf.set_font('Helvetica', '', 10)
    pdf.cell(0, 5, f'Fecha de evaluación: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}', new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 5, f'Perfil: {datos_usuario.get("age_cat")}, {datos_usuario.get("ocupacion")}', new_x="LMARGIN", new_y="NEXT")
    pdf.ln(5)

    # --- FASE 1: CUESTIONARIO ---
    pdf.set_font('Helvetica', 'B', 12)
    pdf.cell(0, 10, 'I. Hallazgos matemáticos (cuestionario de seguimiento)', new_x="LMARGIN", new_y="NEXT")
    pdf.set_font('Helvetica', '', 9)
    
    map_resp = {0: "No / Nunca", 1: "A veces", 2: "Casi siempre"}
    preguntas_pdf = [
        ("Estrés/Relajación", datos_usuario.get('estres')),
        ("Anhedonia/Interés", datos_usuario.get('interes')),
        ("Energía/Herramientas", datos_usuario.get('afrontamiento')),
        ("Estado de ánimo/Tristeza", datos_usuario.get('humor')),
        ("Aislamiento social", datos_usuario.get('debilidad_social')),
        ("Irritabilidad/Frustración", datos_usuario.get('irritabilidad')),
        ("Hábitos/Sueño/Peso", datos_usuario.get('habitos')),
        ("Tiempo de aislamiento", datos_usuario.get('aislamiento')),
        ("Historial clínico previo", datos_usuario.get('historial'))
    ]

    for preg, valor in preguntas_pdf:
        texto_resp = map_resp.get(valor, "Sí, con tratamiento" if valor == 2 else "Sin dato")
        pdf.cell(80, 6, f'- {preg}:', border=0)
        pdf.set_font('Helvetica', 'B', 9)
        pdf.cell(0, 6, f'{texto_resp}', new_x="LMARGIN", new_y="NEXT", border=0)
        pdf.set_font('Helvetica', '', 9)

    pdf.ln(5)
    
    # --- FASE 2: GRÁFICA ---
    pdf.set_font('Helvetica', 'B', 12)
    pdf.cell(0, 10, 'II. Análisis predominante en la conversacion', new_x="LMARGIN", new_y="NEXT")
    pdf.image(ruta_grafica, x=35, w=140)
    pdf.ln(5)

    # --- FASE 3: CONCLUSIÓN Y OBSERVACIONES TÉCNICAS ---
    pdf.set_x(10)
    pdf.set_font('Helvetica', 'B', 12)
    pdf.set_fill_color(240, 240, 240)
    pdf.cell(0, 10, 'III. Conclusión y observaciones técnicas', new_x="LMARGIN", new_y="NEXT", fill=True)
    pdf.ln(4)

    # Variables de porcentajes
    p_normal = promedios[0] * 100
    p_estres = promedios[1] * 100
    p_ansiedad = promedios[2] * 100
    p_depresion = promedios[3] * 100
    p_suicida = promedios[4] * 100

    if p_suicida > 5:
        status_emergencia = "ALERTA CRÍTICA (requiere protocolo de contención inmediato)"
    else:
        status_emergencia = "Negativo (sin riesgo vital inminente detectado en el discurso)"

    emociones_porcentajes = [
        ('Tranquilidad/Normalidad', p_normal), ('Estrés', p_estres), 
        ('Ansiedad', p_ansiedad), ('Depresión', p_depresion), ('Riesgo Suicida', p_suicida)
    ]
    emociones_ordenadas = sorted(emociones_porcentajes, key=lambda x: x[1], reverse=True)
    top1 = emociones_ordenadas[0]
    top2 = emociones_ordenadas[1]

    # ==========================================
    # APARTADO A: RESULTADOS ESPECÍFICOS DEL PACIENTE
    # ==========================================
    pdf.set_font('Helvetica', 'B', 10)
    pdf.cell(0, 8, 'A. Resultados Específicos del Paciente', new_x="LMARGIN", new_y="NEXT")
    pdf.set_font('Helvetica', '', 9)

    resultados_paciente = (
        f"Dictamen general (cuestionario de seguimiento): El paciente es clasificado con {riesgo_global}.\n\n"
        f"Estatus de riesgo autolítico (conversación): {status_emergencia}.\n\n"
        f"Para esta sesión, el modelo detectó que las emociones predominantes expresadas en el discurso fueron "
        f"{top1[0]} ({top1[1]:.1f}%) y {top2[0]} ({top2[1]:.1f}%)."
    )
    pdf.multi_cell(0, 5, resultados_paciente, new_x="LMARGIN", new_y="NEXT")
    pdf.ln(2)

    # Desglose dinámico de lo detectado en este paciente
    hallazgo_normal = f"- Normal ({p_normal:.1f}%): " + ("El paciente mostró un discurso estable con presencia de afecto positivo." if p_normal > 40 else "Los momentos de tranquilidad o emociones agradables fueron poco frecuentes.")
    hallazgo_estres = f"- Estrés ({p_estres:.1f}%): " + ("Se detectó sobrecarga mental, frustración o quejas ambientales significativas en la charla." if p_estres > 25 else "No se detectó sobrecarga situacional aguda en la conversación.")
    hallazgo_ansiedad = f"- Ansiedad ({p_ansiedad:.1f}%): " + ("Se observó sobrepensamiento, miedo al futuro o mención de síntomas físicos." if p_ansiedad > 25 else "El discurso no evidenció ideación ansiosa desbordante.")
    hallazgo_depresion = f"- Depresión ({p_depresion:.1f}%): " + ("ALERTA: Se identificó desesperanza, apatía o tristeza profunda durante la sesión." if p_depresion > 25 else "Sin marcadores fuertes de discurso depresivo.")
    hallazgo_suicida = f"- Riesgo Suicida ({p_suicida:.1f}%): " + ("CRÍTICO: Se detectaron menciones de deseos de muerte o no tener salida." if p_suicida > 5 else "Ausencia de marcadores inminentes de riesgo vital.")

    for hallazgo in [hallazgo_normal, hallazgo_estres, hallazgo_ansiedad, hallazgo_depresion, hallazgo_suicida]:
        pdf.multi_cell(0, 5, hallazgo, new_x="LMARGIN", new_y="NEXT")
    
    pdf.ln(4)

    # ==========================================
    # APARTADO B: OBSERVACIONES TÉCNICAS Y SIMBOLOGÍA
    # ==========================================
    pdf.set_font('Helvetica', 'B', 10)
    pdf.cell(0, 8, 'B. Observaciones Técnicas y Simbología Clínica', new_x="LMARGIN", new_y="NEXT")
    pdf.set_font('Helvetica', '', 9)

    obs_tecnicas = (
        "Es imperativo destacar que este chatbot y su motor NLP fueron desarrollados ÚNICA Y EXCLUSIVAMENTE con el objetivo de "
        "detectar sintomatología y patrones de DEPRESIÓN. Las demás métricas (ansiedad, estrés, normal) se incluyen únicamente "
        "como covariables para entender el contexto clínico del paciente.\n\n"
        "Simbología clínica de la gráfica y extracción de entidades:"
    )
    pdf.multi_cell(0, 5, obs_tecnicas, new_x="LMARGIN", new_y="NEXT")
    pdf.ln(2)
    
    # Explicaciones estáticas (Glosario general)
    simb_normal = "- Normal: Discurso estable y congruente. El paciente denota presencia de afecto positivo (expresiones de tranquilidad, motivación, esperanza), y capacidad de resolución de problemas. No se aprecian alteraciones graves en el estado de ánimo."
    simb_estres = "- Estrés: Carga alostática notable. Detecta sobrecarga mental por responsabilidades cotidianas o indicios de Burnout (ej. quejas sobre falta de tiempo, presiones académicas/laborales)."
    simb_ansiedad = "- Ansiedad: Métrica basada en los criterios del GAD-7. Detecta sobrepensamiento significativo, preocupación excesiva e irracional hacia el futuro, o menciones de síntomas somáticos (taquicardia, falta de aire)."
    simb_depresion = "- Depresión: MÉTRICA OBJETIVO. Basado estrictamente en los criterios del PHQ-9. Identifica lenguaje vinculado a anhedonia (pérdida de interés), desesperanza profunda, culpa excesiva o tristeza clínica (ej. 'no tengo ganas de nada', 'siento que soy un fracaso')."
    simb_suicida = "- Riesgo Suicida: Ideación autolítica. Menciones explícitas o implícitas de deseos de muerte, autolesión, sensación de no tener salida o despedida (ej. 'ya no quiero estar aquí', 'mejor desaparecer')."

    for simb in [simb_normal, simb_estres, simb_ansiedad, simb_depresion, simb_suicida]:
        pdf.multi_cell(0, 5, simb, new_x="LMARGIN", new_y="NEXT")
        pdf.ln(1)

    # NOTA LEGAL
    pdf.ln(3)
    pdf.set_font('Helvetica', 'I', 8)
    pdf.set_text_color(120, 120, 120)
    pdf.multi_cell(0, 4, "Este documento es el resultado de un procesamiento algorítmico de lenguaje natural (NLP) y Machine Learning. No constituye un diagnóstico médico definitivo. Sirve exclusivamente como herramienta de tamizaje (Triage) auxiliar para el profesional de la salud.", new_x="LMARGIN", new_y="NEXT")

    nombre_archivo = f"Reporte_Tecnico_{nombre_usuario.replace(' ', '_')}.pdf"
    
    ruta_pdf_final = os.path.join(carpeta_destino, nombre_archivo)
    
    pdf.output(ruta_pdf_final)
    print(f"[OK] Archivos guardados en carpeta '{carpeta_destino}':")
    print(f"     - Gráfica: {ruta_grafica}")
    print(f"     - Reporte: {ruta_pdf_final}")
if __name__ == "__main__":
    iniciar_aplicacion()

Cargando cerebro del sistema (Esto puede tomar unos segundos)...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 6145.98it/s]
RobertaModel LOAD REPORT from: pysentimiento/robertuito-base-uncased
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



   SISTEMA DE TRIAGE CLÍNICO Y CONTENCIÓN (DEMO TÉSIS)

--- BLOQUE 1: CONTEXTO ---

--- BLOQUE 2: TRIAGE MATEMÁTICO ---

1. En las últimas semanas, ¿has sentido que el estrés te sobrepasa o te cuesta relajarte?
  [0] No / Nunca / Ningún día
  [1] A veces / Regularmente / Varios días
  [2] Sí / Casi siempre / Muy frecuente
  ⚠️ Por favor ingresa un número válido.

2. ¿Has perdido el interés o las ganas de hacer las cosas que antes disfrutabas?
  [0] No / Nunca / Ningún día
  [1] A veces / Regularmente / Varios días
  [2] Sí / Casi siempre / Muy frecuente

3. Cuando enfrentas problemas cotidianos, ¿sientes que te quedas sin energía o herramientas para resolverlos?
  [0] No / Nunca / Ningún día
  [1] A veces / Regularmente / Varios días
  [2] Sí / Casi siempre / Muy frecuente

4. ¿Has notado cambios de humor repentinos, sintiéndote muy triste o sin esperanza?
  [0] No / Nunca / Ningún día
  [1] A veces / Regularmente / Varios días
  [2] Sí / Casi siempre / Muy frecuente

5. ¿Te has senti

##### 

- Cluster 2 es el equivalente al Riesgo Bajo, el Cluster 1 es el Riesgo Alto (rojo crítico), y por descarte, el Cluster 0 es el Moderado.
